# 05 Gradio 接口规范

## 1. 可视化系统概述

### 1.1 系统目标

根据 PDF 第二阶段"Python 综合系统设计"要求，构建一个交互式可视化系统，允许用户：
- 选择和组合数据处理方式
- 选择不同的模型（传统/CNN/无监督）
- 配置损失函数和超参数
- 选择验证方法
- 查看最终分类结果和对比

系统以 Gradio 为前端框架，采用 **5-Tab 布局**。本 Notebook 定义所有回调函数的接口规范，为后续可视化系统开发提供统一的对接标准。

### 1.2 5-Tab 布局设计

```
┌─────────────────────────────────────────────────┐
│  Tab 1: 数据处理  │ Tab 2: 模型选择 │ Tab 3: 损失函数 │
│  Tab 4: 参数优化  │ Tab 5: 模型验证               │
└─────────────────────────────────────────────────┘
```

| Tab | 名称 | 对应回调函数 | 功能 |
|-----|------|-------------|------|
| 1 | 数据处理 | `data_processing_handler()` | 选择划分策略、预处理方法、特征类型 |
| 2 | 模型选择 | `traditional_model_handler()` / `cnn_handler()` / `unsupervised_handler()` | 选择模型类型并配置 |
| 3 | 损失函数 | `cnn_handler(loss_fn=...)` | 选择损失函数（仅 CNN） |
| 4 | 参数优化 | 各 handler 的 `params` 参数 | 配置超参数 |
| 5 | 模型验证 | `compare_all_models_handler()` | 综合对比所有结果 |

## 2. 接口规范总览

| 接口函数 | 所属 Notebook | 输入参数 | 输出类型 |
|----------|:--:|------|------|
| `data_processing_handler` | 01 | split_method, split_ratio, k_folds, use_stratify, preprocessing, features | dict (comparison_table, best_pipeline, plots) |
| `traditional_model_handler` | 02 | model_type, preprocessing, features, model_params, cv_folds | dict (metrics, confusion_matrix, roc_curve, cv_results) |
| `cnn_handler` | 03 | preprocessing, loss_fn, focal_alpha/gamma, optimizer, lr, dropout, batch_size, epochs | dict (metrics, train_history, confusion_matrix, roc_curve, best_hyperparams) |
| `unsupervised_handler` | 04 | method, n_clusters, feature_set, method_params | dict (labels, internal_metrics, external_metrics, cluster_plot) |
| `compare_all_models_handler` | 05 | model_configs, validation_method | dict (comparison_table, roc_curves, summary) |

## 3. 数据处理接口规范

### `data_processing_handler`

**来源**：`01_数据处理与特征工程.ipynb` 第 10 节

```
def data_processing_handler(
    split_method: str = "holdout",   # "holdout" | "kfold"
    split_ratio: float = 0.7,        # 留出法训练集比例
    k_folds: int = 5,                # K折数
    use_stratify: bool = True,       # 是否分层抽样
    preprocessing: list = None,      # ["none","clahe","gaussian","median","clahe+gaussian","clahe+median"]
    features: list = None,           # ["hog","lbp","glcm","edge_density"]
    max_samples: int = 2000,         # 样本上限
    random_seed: int = 42,           # 随机种子
) -> dict                            # {comparison_table, best_pipeline, best_features, ...}
```

## 4. 传统模型接口规范

### `traditional_model_handler`

**来源**：`02_传统监督学习对比.ipynb` 第 14 节

```
def traditional_model_handler(
    model_type: str = "random_forest",  # "decision_tree"|"svm"|"naive_bayes"|"random_forest"|"logistic_regression"|"xgboost"|"lightgbm"
    preprocessing: list = None,         # 预处理方法
    features: list = None,              # 特征类型
    model_params: dict = None,          # {"n_estimators":100,"max_depth":20,...}
    cv_folds: int = 5,                  # 交叉验证折数
    max_samples: int = 2000,            # 样本上限
    random_seed: int = 42,              # 随机种子
) -> dict                               # {model_name, metrics, confusion_matrix, roc_curve, cv_results, feature_importance, training_time}
```

### 各模型支持的参数

| 模型 | 关键参数 |
|------|------|
| 决策树 | max_depth, criterion, min_samples_split |
| SVM | kernel, C, gamma |
| 朴素贝叶斯 | var_smoothing |
| 随机森林 | n_estimators, max_depth, min_samples_split |
| 逻辑回归 | C, penalty, solver |
| XGBoost | n_estimators, max_depth, learning_rate, subsample |
| LightGBM | n_estimators, max_depth, learning_rate, num_leaves |

## 5. CNN 接口规范

### `cnn_handler`

**来源**：`03_深度学习对比.ipynb` 第 11 节

这是可视化系统中参数最多的核心接口，覆盖模型选择、损失函数、参数优化三个 Tab。

```
def cnn_handler(
    preprocessing: list = None,              # 预处理管线
    loss_fn: str = "cross_entropy",          # "cross_entropy"|"focal"|"label_smoothing"|"dice"
    focal_alpha: float = 0.25,               # Focal Loss alpha (0~1)
    focal_gamma: float = 2.0,                # Focal Loss gamma (0~5)
    label_smoothing_epsilon: float = 0.1,    # Label Smoothing epsilon
    optimizer_name: str = "adam",            # "adam"|"sgd"
    learning_rate: float = 1e-3,             # 学习率
    dropout_rate: float = 0.5,               # Dropout 比例
    batch_size: int = 64,                    # 批量大小
    epochs: int = 30,                        # 最大训练轮数
    early_stopping_patience: int = 10,       # 早停耐心值
    max_samples: int = 4000,                 # 样本上限
    random_seed: int = 42,                   # 随机种子
) -> dict                                    # {metrics, train_history, confusion_matrix, roc_curve, best_hyperparams, model_weights_path}
```

### 返回值详情

| 字段 | 类型 | 说明 |
|------|------|------|
| metrics | dict | 测试集评估指标 (accuracy, precision, recall, f1, roc_auc) |
| train_history | dict | 训练曲线数据 {train_loss, val_loss, train_acc, val_acc} |
| confusion_matrix | plt.Figure | 混淆矩阵图 |
| roc_curve | plt.Figure | ROC 曲线图 |
| best_hyperparams | dict | 实际使用的超参数组合 |
| model_weights_path | str | 模型权重保存路径 |

## 6. 无监督接口规范

### `unsupervised_handler`

**来源**：`04_无监督学习对比.ipynb` 第 5 节

```
def unsupervised_handler(
    method: str = "kmeans",            # "kmeans"|"gmm"|"dbscan"|"agglomerative"|"spectral"
    n_clusters: int = 2,               # 聚类数量
    feature_set: list = None,          # 特征子集
    method_params: dict = None,        # 方法特定参数
    max_samples: int = 2000,           # 样本上限
    random_seed: int = 42,             # 随机种子
) -> dict                              # {labels, internal_metrics, external_metrics, cluster_plot, method_summary}
```

### 各方法默认参数

| 方法 | 默认参数 | 特殊说明 |
|------|------|------|
| K-Means | n_clusters=2, random_state=42, n_init='auto' | — |
| GMM | n_components=2, random_state=42 | 支持概率输出 |
| DBSCAN | eps=0.5, min_samples=5 | 噪声点标签=-1 |
| 层次聚类 | n_clusters=2, linkage='ward' | — |
| 谱聚类 | n_clusters=2, affinity='rbf', random_state=42 | 对非凸形状数据有效 |

## 7. 接口组合示例

### 端到端对比实验流程

```python
# 示例 1：对比不同预处理对随机森林的影响
data_result = data_processing_handler(
    split_method="holdout", split_ratio=0.7,
    preprocessing=["clahe", "median"],
    features=["hog", "lbp", "glcm", "edge_density"]
)
model_result = traditional_model_handler(
    model_type="random_forest",
    model_params={"n_estimators": 100, "max_depth": 20}
)

# 示例 2：CNN 完整训练并对比损失函数
cnn_result_ce = cnn_handler(
    preprocessing=["clahe", "median"],
    loss_fn="cross_entropy",
    learning_rate=1e-3, epochs=30
)
cnn_result_focal = cnn_handler(
    preprocessing=["clahe", "median"],
    loss_fn="focal", focal_alpha=0.25, focal_gamma=2.0,
    learning_rate=1e-3, epochs=30
)

# 示例 3：无监督聚类探索
unsup_result = unsupervised_handler(
    method="kmeans", n_clusters=2,
    feature_set=["hog", "lbp", "glcm", "edge_density"]
)
```

## 8. 跨模型对比接口

### `compare_all_models_handler`

统一的跨模型对比入口，用于 Tab 5 "模型验证"。

```
def compare_all_models_handler(
    model_configs: list[dict],         # [{"type":"rf","params":{...}},{"type":"cnn","params":{...}},...]
    data_config: dict = None,          # 数据处理配置（若为None则使用默认）
    validation_method: str = "holdout", # "holdout"|"kfold"
) -> dict:
    """统一跨模型对比接口

    依次运行所有配置的模型，收集结果并生成综合对比表。

    Parameters
    ----------
    model_configs : list[dict]
        每个元素为 {"type": str, "params": dict}。
        type 可选: "rf","svm","nb","dt","lr","xgboost","lightgbm","cnn","kmeans","gmm","dbscan".
    data_config : dict or None
        数据处理配置。None 则使用默认管线。
    validation_method : str
        验证方法。

    Returns
    -------
    dict
        {
            "comparison_table": pd.DataFrame,  # 所有模型的指标对比
            "roc_curves": plt.Figure,          # ROC曲线叠加图
            "best_model": str,                 # 最优模型名称
            "best_metrics": dict,              # 最优模型的指标
            "summary_text": str,               # 文本摘要
        }
    """
    # 可视化系统阶段实现
    pass
```

## 9. 接口实现状态

| 接口函数 | 状态 | 实现阶段 |
|----------|:--:|------|
| `data_processing_handler` | 🔶 接口已定义 | 可视化系统开发 |
| `traditional_model_handler` | 🔶 接口已定义 | 可视化系统开发 |
| `cnn_handler` | 🔶 接口已定义 | 可视化系统开发 |
| `unsupervised_handler` | 🔶 接口已定义 | 可视化系统开发 |
| `compare_all_models_handler` | 🔶 接口已定义 | 可视化系统开发 |

> 🔶 = 接口规范已完整定义（函数签名 + 参数说明 + 返回值格式），具体实现将在可视化系统开发阶段完成。

### 下一步

1. 在 `pyproject.toml` 或 `requirements.txt` 中添加 `gradio` 依赖
2. 创建 `src/gradio_app.py` 实现具体的 Gradio Blocks 界面
3. 根据本规范实现每个回调函数的内部逻辑
4. 测试并优化界面交互体验